# [통신 실습 3] PCM – 표본화·양자화·부호화

**PCM(Pulse Code Modulation)** 은 아날로그 신호를 디지털 비트열로 바꾸는 가장 기본적인 방법입니다. 전화망, CD 음원 등에 쓰입니다.

| 단계 | 내용 |
|---|---|
| ① 표본화(Sampling) | 일정한 시간 간격으로 값을 뽑음 (나이퀴스트: 최고 주파수의 2배 이상) |
| ② 양자화(Quantization) | 뽑은 값을 정해진 단계(레벨) 중 가까운 값으로 맞춤 |
| ③ 부호화(Encoding) | 각 레벨 번호를 이진수(비트)로 변환 |

3비트를 쓰면 $2^3 = 8$단계로 표현합니다.

> 위에서부터 차례로 `Shift + Enter`로 실행하세요.

## 0. 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False   # 그래프의 마이너스(-) 기호 깨짐 방지
print("한글 폰트 설정 완료")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. 아날로그 신호 생성

In [ ]:
# 1. 아날로그 신호 생성
t_continuous = np.linspace(0, 0.01, 1000)
m_t = np.sin(2 * np.pi * 100 * t_continuous) # 100Hz 아날로그 신호

## 2. 표본화 (Sampling) - 초당 800회 샘플 추출

In [ ]:
# 2. 표본화 (Sampling) - 초당 800회 샘플 추출
fs = 800
t_sampled = np.arange(0, 0.01, 1 / fs)   # 정확히 1/fs 간격으로 표본화
sampled_signal = np.sin(2 * np.pi * 100 * t_sampled)

## 3. 양자화 (Quantization) - 3비트 (8단계 레벨)

In [ ]:
# 3. 양자화 (Quantization) - 3비트 (8단계 레벨)
bits = 3
levels = 2**bits
min_val, max_val = -1.0, 1.0
step_size = (max_val - min_val) / levels

# 아날로그 값을 가장 가까운 양자화 대표값으로 매핑
quantized_idx = np.floor((sampled_signal - min_val) / step_size)
quantized_idx = np.clip(quantized_idx, 0, levels - 1)
quantized_signal = min_val + (quantized_idx + 0.5) * step_size

## 4. 시각화

In [ ]:
# 4. 시각화
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(t_continuous * 1000, m_t, 'b', label="연속 아날로그 신호")
plt.stem(t_sampled * 1000, sampled_signal, 'r', markerfmt='ro', basefmt=" ", label="표본화(Sampling)")
plt.title("1. 표본화 과정 (Sampling)")
plt.xlabel("시간 (ms)")
plt.ylabel("진폭")
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t_continuous * 1000, m_t, 'b--', alpha=0.4)
plt.step(t_sampled * 1000, quantized_signal, 'g', where='mid', label=f"{bits}비트 ({levels}단계) 양자화")
plt.title("2. 양자화 과정 (Quantization Step)")
plt.xlabel("시간 (ms)")
plt.ylabel("진폭 레벨")
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 3)
# 비트 부호화 데이터 출력 (예시 표시)
bit_stream = [bin(int(idx))[2:].zfill(bits) for idx in quantized_idx[:8]]
plt.text(0.1, 0.5, f"샘플링된 비트 스트림 예시 (처음 8개 샘플):\n{ ' -> '.join(bit_stream) }", 
         fontsize=12, bbox=dict(facecolor='yellow', alpha=0.5))
plt.axis('off')
plt.title("3. 부호화 (Encoding: 아날로그 -> 이진 비트열 변환)")

plt.tight_layout()
plt.show()

### 원본 코드에서 수정한 부분
- 표본화 간격이 정확히 1/fs(=1.25ms)가 되도록 `np.linspace` 대신 `np.arange`를 사용했습니다.
- 양자화 구간 번호를 `np.round` 대신 `np.floor`로 구해야 각 구간의 중앙값(대표값)과 맞습니다.

## 직접 해 보기
- `bits`를 2, 4, 6으로 바꿔 양자화 계단이 어떻게 변하는지 보세요.
- 표본화 주파수 `fs`를 150Hz(나이퀴스트 미만)로 낮추면 어떻게 되나요?
- 값을 바꾼 뒤에는 바뀐 셀부터 아래로 다시 실행하세요.